# PediTrack RAG - GPU Accelerated Dataset Ingestion

This notebook ingests all pediatric medical datasets into a FAISS vector store using GPU acceleration.

**Datasets included:**
- HealthCareMagic (24,855 docs)
- Medical QA Pediatric (6,500 docs)
- Symptom Checker (5,002 docs)
- PediatricsMQA (3,417 docs)

**Total: ~40,000 documents**

## Step 1: Setup Environment

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ No GPU detected. Make sure to enable GPU in Runtime > Change runtime type")

In [ ]:
# Install required packages
!pip install -q sentence-transformers faiss-gpu numpy tqdm

## Step 2: Upload Processed Data

Upload the `processed_data.zip` file from your local machine.

In [ ]:
from google.colab import files
import zipfile

# Upload the zip file
print("Please upload processed_data.zip")
uploaded = files.upload()

# Extract
with zipfile.ZipFile('processed_data.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("✓ Data extracted")

## Step 3: Load Embedding Model (GPU)

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

# Load model on GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on {device}...")

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
embedding_dim = 384

print(f"✓ Model loaded on {device}")
print(f"Embedding dimension: {embedding_dim}")

## Step 4: Initialize FAISS Vector Store

In [ ]:
import faiss
import numpy as np

# Create FAISS index
index = faiss.IndexFlatL2(embedding_dim)
metadata_list = []

print(f"✓ FAISS index created (dimension: {embedding_dim})")

## Step 5: Ingest All Datasets

In [ ]:
import json
from pathlib import Path
from tqdm.notebook import tqdm

# Find all processed files
processed_files = list(Path('processed').glob('*_processed.json'))
processed_files = [f for f in processed_files if 'combined' not in f.name.lower()]

print(f"Found {len(processed_files)} dataset files:")
for f in processed_files:
    print(f"  - {f.name}")

total_docs = 0
batch_size = 100

for processed_file in processed_files:
    print(f"\n📥 Processing: {processed_file.name}")
    
    # Load documents
    with open(processed_file, 'r', encoding='utf-8') as f:
        documents = json.load(f)
    
    print(f"  Documents: {len(documents):,}")
    
    # Process in batches
    for i in tqdm(range(0, len(documents), batch_size), desc="  Embedding"):
        batch = documents[i:i + batch_size]
        
        # Extract texts
        texts = [doc['text'] for doc in batch]
        
        # Generate embeddings on GPU
        embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
        
        # Add to FAISS
        index.add(embeddings.astype('float32'))
        
        # Store metadata
        for doc in batch:
            metadata_list.append({
                'text': doc['text'],
                'source': doc['source'],
                'metadata': doc.get('metadata', {})
            })
    
    total_docs += len(documents)
    print(f"  ✓ Ingested {len(documents):,} documents")

print(f"\n✅ Total documents ingested: {total_docs:,}")
print(f"Vector store size: {index.ntotal:,}")

## Step 6: Save Vector Store

In [ ]:
import json

# Save FAISS index
faiss.write_index(index, 'faiss_index')
print("✓ FAISS index saved")

# Save metadata
with open('metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata_list, f, ensure_ascii=False)
print("✓ Metadata saved")

print(f"\nFiles created:")
print(f"  - faiss_index ({index.ntotal:,} vectors)")
print(f"  - metadata.json ({len(metadata_list):,} entries)")

## Step 7: Download Vector Store

In [ ]:
from google.colab import files
import zipfile

# Create zip file
with zipfile.ZipFile('vector_store.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('faiss_index')
    zipf.write('metadata.json')

print("✓ Vector store packaged")

# Download
files.download('vector_store.zip')
print("✓ Download started")

## Step 8: Test Retrieval (Optional)

In [ ]:
# Test query
query = "What are the symptoms of fever in children?"
query_embedding = model.encode([query], convert_to_numpy=True)

# Search
k = 3
distances, indices = index.search(query_embedding.astype('float32'), k)

print(f"Query: {query}\n")
print(f"Top {k} results:\n")

for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
    print(f"{i+1}. Score: {1/(1+dist):.3f}")
    print(f"   Source: {metadata_list[idx]['source']}")
    print(f"   Text: {metadata_list[idx]['text'][:200]}...")
    print()

---

## 🎉 Complete!

Your vector store has been created with GPU acceleration and is ready to download.

**Next steps:**
1. Download `vector_store.zip`
2. Extract to your local `vector_store/` directory
3. Start your RAG service

**Stats:**
- Total documents: ~40,000
- Embedding dimension: 384
- Model: sentence-transformers/all-MiniLM-L6-v2
- Processing: GPU accelerated ⚡